# GastroVision — video re-evaluation (no training; ~10 min on GPU, 30–45 min on CPU)

Re-scores the **held-out val/test videos and your 3 external clips** with the *exported ONNX model and the app's own preprocessing + verdict code*,
at a fixed 1 frame/s over each whole video (first 240 s). Writes `video_frame_probs.csv` (small) — send that file back so the verdict rules can be tuned on real video.

**Setup (Kaggle):** Accelerator: GPU T4 is faster, CPU also works · Internet **On** · Add Data:
1. the same HyperKvasir dataset (with `labeled-videos`) as the training run,
2. the training notebook's **Output** (Add Data → Notebook Output Files → pick the training run) *or* a dataset holding `gastrovision_model_bundle.zip`.
Then *Run All*.

In [ ]:
import os, sys, re, json, zipfile, shutil, time, subprocess
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, cv2
INPUT = os.environ.get("GV_INPUT", "/kaggle/input"); WORK = os.environ.get("GV_WORK", "/kaggle/working")
try:
    import onnxruntime as ort
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime"], check=False); import onnxruntime as ort

def find(name):
    for dp, _, fns in os.walk(INPUT):
        if name in fns: return os.path.join(dp, name)
BUNDLE = Path(WORK) / "bundle"; BUNDLE.mkdir(parents=True, exist_ok=True)
z = find("gastrovision_model_bundle.zip")
if z: zipfile.ZipFile(z).extractall(BUNDLE); print("bundle unzipped from", z)
else:
    for n in ("gastrovision_cnn.onnx", "class_names.json", "preprocess.json", "gastrovision_preprocess.py", "splits.csv", "metrics.json"):
        p = find(n)
        if p: shutil.copy(p, BUNDLE / n)
need = ["gastrovision_cnn.onnx", "class_names.json", "preprocess.json", "gastrovision_preprocess.py", "splits.csv"]
miss = [n for n in need if not (BUNDLE / n).exists()]
assert not miss, f"missing {miss}: add the training notebook's output (or the bundle zip) via Add Data"
sys.path.insert(0, str(BUNDLE)); import gastrovision_preprocess as gp
cfg = gp.load_cfg(str(BUNDLE)); CLASSES = json.load(open(BUNDLE / "class_names.json")); T = float(cfg["temperature"]); THR = float(cfg["conf_threshold"])
prov = [p for p in ("CUDAExecutionProvider", "CPUExecutionProvider") if p in ort.get_available_providers()]
sess = ort.InferenceSession(str(BUNDLE / "gastrovision_cnn.onnx"), providers=prov); inp = sess.get_inputs()[0].name
print("classes:", CLASSES, "| T =", round(T, 3), "| conf threshold =", THR, "| provider:", prov[0])
def probs_bgr(frames):
    out = []
    for i in range(0, len(frames), 16):
        lg = sess.run(None, {inp: gp.preprocess_batch_bgr(frames[i:i + 16], cfg)})[0] / T
        lg = lg - lg.max(1, keepdims=True); e = np.exp(lg); out.append(e / e.sum(1, keepdims=True))
    return np.concatenate(out) if out else np.zeros((0, len(CLASSES)), np.float32)

In [ ]:
%%writefile gastrovision_video_scoring.py
"""
scoring/video_scoring.py
========================
Turns per-frame probabilities into ONE verdict for the whole video.

Why the old rule was unreliable
-------------------------------
The old code returned the class with the highest *average confidence*.  A single frame the
model was 99 % sure about (on a class seen once) beat a class seen in hundreds of frames.

New rule (all thresholds are parameters):
  1. smooth the per-class probability curves over time (median filter, 5 frames);
  2. a frame "votes" for a class only if it is the arg-max, is not the `other` class and is
     at least `conf_threshold` confident (calibrated);
  3. a class counts as DETECTED only if it has a run of >= `min_run` consecutive voting frames
     (a real lesion stays in view for a moment; single-frame noise does not);
  4. the headline verdict is taken from the most clinically important tier that has a detection:
         pathology  (polyps, esophagitis, ulcerative-colitis)
         > procedure (dyed-lifted-polyps, dyed-resection-margins)
         > landmark (normal-cecum, normal-pylorus, normal-z-line)
     and within a tier the class with most voting frames wins;
  5. nothing detected -> "no-finding".

The full list of findings (with timing) is always returned so the UI can show all of them.
"""
import json
import math
import os
from typing import Dict, List, Optional, Tuple

import numpy as np

PATHOLOGY = ("polyps", "esophagitis", "ulcerative-colitis")
PROCEDURE = ("dyed-lifted-polyps", "dyed-resection-margins")
LANDMARK = ("normal-cecum", "normal-pylorus", "normal-z-line")
TIER_ORDER = ["pathology", "procedure", "landmark"]
NO_FINDING = "no-finding"


def tier_of(cls: str) -> str:
    if cls in PATHOLOGY:
        return "pathology"
    if cls in PROCEDURE:
        return "procedure"
    if cls in LANDMARK:
        return "landmark"
    return "other"


def _median_smooth(P: np.ndarray, window: int = 5) -> np.ndarray:
    if window <= 1 or len(P) < 3:
        return P
    window = min(window, len(P) if len(P) % 2 == 1 else len(P) - 1)
    pad = window // 2
    Pp = np.pad(P, ((pad, pad), (0, 0)), mode="edge")
    win = np.lib.stride_tricks.sliding_window_view(Pp, window, axis=0)   # [n, K, window]
    return np.median(win, axis=-1)


def _longest_run(mask: np.ndarray) -> int:
    best = cur = 0
    for v in mask:
        cur = cur + 1 if v else 0
        best = max(best, cur)
    return best


class video_scoring:
    # ---------------------------------------------------------------- loading
    @staticmethod
    def _video_dir(video_name: str) -> str:
        backend_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
        return os.path.join(backend_dir, "outputs", video_name)

    def load_frame_scores(self, video_name: str) -> Tuple[List[str], List[str], np.ndarray]:
        names, classes, rows = [], None, []
        with open(os.path.join(self._video_dir(video_name), "frame_scores.jsonl"), "r", encoding="utf-8") as f:
            for line in f:
                d = json.loads(line)
                if classes is None:
                    classes = list(d["probabilities"].keys())
                names.append(d["frame"])
                rows.append([d["probabilities"][c] for c in classes])
        return names, classes or [], np.asarray(rows, np.float32).reshape(len(names), len(classes or []))

    # ---------------------------------------------------------------- core
    def summarize_probs(self, names: List[str], classes: List[str], P: np.ndarray, conf_threshold: float = 0.7,
                        min_run: int = 3, smooth: int = 5, other_class: Optional[str] = "other") -> Dict:
        n = len(names)
        if n == 0:
            return {"label": NO_FINDING, "confidence": 0.0, "findings": [], "n_frames": 0, "n_rejected": 0,
                    "reason": "no usable frames"}
        Ps = _median_smooth(P, smooth)
        pred, conf = Ps.argmax(1), Ps.max(1)
        other_idx = classes.index(other_class) if other_class in classes else -1
        voting = (conf >= conf_threshold) & (pred != other_idx)
        findings = []
        for k, c in enumerate(classes):
            if k == other_idx:
                continue
            m = voting & (pred == k)
            run = _longest_run(m)
            if run < min_run:
                continue
            idx = np.where(m)[0]
            best = int(idx[np.argmax(P[idx, k])])
            findings.append({
                "class": c, "tier": tier_of(c), "frames": int(m.sum()), "longest_run": int(run),
                "mean_confidence": float(Ps[idx, k].mean()), "peak_confidence": float(P[best, k]),
                "first_frame": names[int(idx[0])], "best_frame": names[best], "best_index": best,
                "fraction_of_video": float(m.sum() / n),
            })
        tier_rank = {t: i for i, t in enumerate(TIER_ORDER + ["other"])}
        findings.sort(key=lambda f: (tier_rank[f["tier"]], -f["frames"], -f["mean_confidence"]))
        n_rejected = int((~voting).sum())
        if findings:
            top = findings[0]
            label, confidence = top["class"], top["mean_confidence"]
        else:
            # Nothing consistent was found.  `confidence` is 0 on purpose: it used to be the fraction of
            # rejected frames, which the UI showed as a misleading "100 % confidence".
            label, confidence = NO_FINDING, 0.0
        out = {"label": label, "confidence": float(confidence), "findings": findings, "n_frames": n,
               "n_rejected": n_rejected, "uncertain_fraction": float(n_rejected / n),
               "conf_threshold": conf_threshold, "min_run": min_run}
        if label == NO_FINDING:
            out["note"] = ("No target finding was seen consistently. This does NOT mean the video is normal: "
                           "the model only knows 8 findings plus 'other', and frames that look different from "
                           "its training images are rejected.")
        return out

    def summarize(self, video_name: str, conf_threshold: float = 0.7, min_run: int = 3, smooth: int = 5,
                  other_class: Optional[str] = "other") -> Dict:
        names, classes, P = self.load_frame_scores(video_name)
        out = self.summarize_probs(names, classes, P, conf_threshold, min_run, smooth, other_class)
        with open(os.path.join(self._video_dir(video_name), "video_summary.json"), "w", encoding="utf-8") as f:
            json.dump(out, f, indent=2)
        return out

    # ---------------------------------------------------------------- helpers
    def select_heatmap_frames(self, summary: Dict, video_name: str, top_n: int = 5, min_gap: int = 6) -> List[Dict]:
        """Pick up to `top_n` well-spread, high-confidence frames, round-robin over the detected findings."""
        names, classes, P = self.load_frame_scores(video_name)
        picked, per_class = [], []
        for f in summary.get("findings", []):
            k = classes.index(f["class"])
            order = [int(i) for i in np.argsort(-P[:, k]) if P[i, k] >= summary.get("conf_threshold", 0.7)]
            chosen = []
            for i in order:
                if all(abs(i - j) >= min_gap for j in chosen):
                    chosen.append(i)
            per_class.append((f["class"], chosen))
        r = 0
        while len(picked) < top_n and any(r < len(c) for _, c in per_class):
            for cls, c in per_class:
                if r < len(c) and len(picked) < top_n:
                    i = c[r]
                    k = classes.index(cls)
                    picked.append({"frame": names[i], "label": cls, "confidence": float(P[i, k]), "rank": len(picked) + 1})
            r += 1
        return picked

    # ---------------------------------------------------------------- backwards-compatible API
    def average_confidence(self, video_name: str, conf_threshold: float = 0.7):
        s = self.summarize(video_name, conf_threshold)
        return s["label"], s["confidence"]

In [ ]:
from gastrovision_video_scoring import video_scoring
VID_EXT = {".avi", ".mp4", ".mov", ".mkv", ".mpg", ".mpeg", ".wmv", ".m4v"}
HOLDOUT = ["16e7e461-a9a7-4b04-8d2c-155de3eb1c90", "29ce8d04-c184-42fa-ac05-322d3a2d37bd", "2a200509-e1d3-46bd-8ae7-7fb1a8491709"]
EXTRA = {"cecum": "normal-cecum", "pylorus": "normal-pylorus", "z-line": "normal-z-line"}
def truth_from_path(p):
    for part in reversed(Path(p).parts[:-1]):
        n = part.lower()
        if n in CLASSES: return n
        if n in EXTRA: return EXTRA[n]
        if n.startswith("esophagitis"): return "esophagitis"
        if n.startswith("ulcerative-colitis"): return "ulcerative-colitis"
    return None
stem_path = {}
for dp, _, fns in os.walk(INPUT):
    for f in fns:
        if Path(f).suffix.lower() in VID_EXT and "unlabeled" not in dp.lower(): stem_path.setdefault(Path(f).stem, os.path.join(dp, f))
sp = pd.read_csv(BUNDLE / "splits.csv")
vv = sp[sp.vid.notna() & sp.split.isin(["val", "test"])].drop_duplicates("vid")
jobs = []
for _, r in vv.iterrows():
    p = stem_path.get(Path(r.vid).stem, r.vid)
    if os.path.exists(p): jobs.append((r.split, Path(r.vid).stem, r.cls, p))
    else: print("video not found:", r.vid)
for h in HOLDOUT:
    if h in stem_path: jobs.append(("external", h, truth_from_path(stem_path[h]), stem_path[h]))
    else: print("external clip not found in dataset:", h)
print(f"{len(jobs)} videos to score:", Counter(j[0] for j in jobs))

In [ ]:
FPS, MAX_FRAMES = 1.0, 240
def read_frames(job):
    path = job[3]; frames = []
    cap = cv2.VideoCapture(path); fps = cap.get(cv2.CAP_PROP_FPS); fps = fps if fps and 1 < fps < 240 else 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0); step = max(1, int(round(fps / FPS)))
    want = set(range(step // 2, total, step)) if total else set(); want = set(sorted(want)[:MAX_FRAMES]); last = max(want) if want else -1; k = 0
    while k <= last:
        if not cap.grab(): break
        if k in want:
            ok, fr = cap.retrieve()
            if ok: frames.append(fr)
        k += 1
    cap.release()
    if not frames:
        out = Path(WORK) / "tmp_frames" / job[1]; out.mkdir(parents=True, exist_ok=True)
        subprocess.run(["ffmpeg", "-v", "error", "-i", path, "-vf", f"fps={FPS}", "-frames:v", str(MAX_FRAMES), "-q:v", "2", str(out / "f%05d.jpg")], check=False)
        frames = [cv2.imread(str(f)) for f in sorted(out.glob("*.jpg"))]
    return frames
rows = []; t0 = time.time()
for c in range(0, len(jobs), 8):
    chunk = jobs[c:c + 8]
    with ThreadPoolExecutor(4) as ex: fr_list = list(ex.map(read_frames, chunk))
    for job, frames in zip(chunk, fr_list):
        if not frames: print("  no frames for", job[1]); continue
        P = probs_bgr(frames)
        for k, p in enumerate(P): rows.append([job[0], job[1], job[2], k] + [round(float(x), 5) for x in p])
    print(f"  {min(c + 8, len(jobs))}/{len(jobs)} videos | {len(rows)} frames | {time.time() - t0:.0f}s", flush=True)
FP = pd.DataFrame(rows, columns=["split", "vid", "truth", "t"] + CLASSES)
FP.to_csv(Path(WORK) / "video_frame_probs.csv", index=False); print("saved video_frame_probs.csv", FP.shape)

In [ ]:
vsc = video_scoring(); OTHER = "other" if "other" in CLASSES else None
def verdicts(FP, thr=THR, min_run=2, smooth=3):
    out = []
    for (sp_, v), g in FP.groupby(["split", "vid"]):
        g = g.sort_values("t"); P = g[CLASSES].to_numpy(np.float32)
        s = vsc.summarize_probs([f"{int(t):05d}" for t in g.t], CLASSES, P, conf_threshold=thr, min_run=min_run, smooth=smooth, other_class=OTHER)
        truth = g.truth.iloc[0]; det = [f["class"] for f in s["findings"]]
        hit = (len(det) == 0) if truth == "other" else (truth in det)
        head = (s["label"] == "no-finding") if truth == "other" else (s["label"] == truth)
        out.append(dict(split=sp_, video=v[:8], truth=truth, n=len(g), headline=s["label"], detected=det, hit=bool(hit), headline_ok=bool(head),
                        false_findings=len([d for d in det if d != truth]), frame_argmax=dict(Counter(CLASSES[i] for i in P.argmax(1)).most_common(3))))
    return pd.DataFrame(out)
R = verdicts(FP)
for sp_ in ("test", "val"):
    r = R[R.split == sp_]
    if len(r):
        print(f"\n=== {sp_.upper()} videos: {len(r)} | labelled finding detected / correct 'nothing': {100 * r.hit.mean():.0f}% | headline correct: {100 * r.headline_ok.mean():.0f}% | false extra findings per video: {r.false_findings.mean():.2f}")
        print(r.groupby("truth").agg(videos=("hit", "size"), hit=("hit", "sum"), headline_ok=("headline_ok", "sum"), false_findings=("false_findings", "sum")).to_string())
print("\n=== EXTERNAL clips (never trained on)")
print(R[R.split == "external"][["video", "truth", "n", "headline", "detected", "frame_argmax"]].to_string(index=False))
print("\nNow download /kaggle/working/video_frame_probs.csv and send it back.")